# Lab 2 — Trader project (market-data MCP)

This lab is the trader project.

- MCP server file: `market_data_server.py`
- Free data path: `get_stooq_quote` (no key)
- Optional key path: `get_polygon_previous_close` with `POLYGON_API_KEY`

Goal: run a trader-style agent that chooses tools to answer market prompts.

In [18]:
from pathlib import Path

from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio

load_dotenv(override=True)

EXERCISE_DIR = Path.cwd().resolve()

REPO_ROOT = EXERCISE_DIR.parent.parent.parent

market_mcp_params = {
    "command": "uv",
    "args": ["run", str(EXERCISE_DIR / "market_data_server.py")],
    "cwd": str(REPO_ROOT),
}

### Run trader prompt with free Stooq data

In [19]:
async with MCPServerStdio(params=market_mcp_params, client_session_timeout_seconds=30) as server:
    market_tools = await server.list_tools()

market_tools

[Tool(name='get_stooq_quote', title=None, description='Latest daily OHLC from Stooq (free, no API key).\n\n    Args:\n        symbol: Ticker in Stooq form, e.g. ``aapl.us``, ``msft.us``, ``gbpusd`` for FX.\n\n    Returns:\n        Human-readable line with date, open, high, low, close, volume (if available).\n    ', inputSchema={'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'get_stooq_quoteArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'get_stooq_quoteOutput', 'type': 'object'}, icons=None, annotations=None, meta=None),
 Tool(name='get_polygon_previous_close', title=None, description='Previous close via Polygon REST API. Requires ``POLYGON_API_KEY`` (free or paid plan).\n\n    Args:\n        symbol: US stock ticker, e.g. ``AAPL``.\n\n    Returns:\n        Previous close price or an error / hint string.\n    ', inputSchema={'properties': {'sy

In [20]:
from IPython.display import Markdown, display

request = "Use get_stooq_quote for aapl.us and summarize the latest daily quote for a cautious trader in 3 bullet points."

async with MCPServerStdio(params=market_mcp_params, client_session_timeout_seconds=30) as mcp_server:
    trader_agent = Agent(
        name="trader_assistant",
        instructions=(
            "You are a careful market assistant. Use MCP tools for prices, "
            "state if data is delayed, and avoid financial-advice certainty."
        ),
        model="gpt-4o-mini",
        mcp_servers=[mcp_server],
    )
    with trace("adeyemi_lab2_trader"):
        result = await Runner.run(trader_agent, request)

display(Markdown(result.final_output))

Here’s the latest daily quote for AAPL (Apple Inc.):

- **Closing Price**: The stock closed at $246.63.
- **Price Range**: It experienced a high of $250.87 and a low of $245.51 during the trading session.
- **Volume**: Approximately 39,446,213 shares were traded.

(Note: This data is current as of March 30, 2026.)

In [21]:
# Optional: test Polygon path if POLYGON_API_KEY is set
request_polygon = "Use get_polygon_previous_close for AAPL and explain what this metric means in one sentence."

async with MCPServerStdio(params=market_mcp_params, client_session_timeout_seconds=30) as mcp_server:
    trader_agent = Agent(
        name="trader_assistant_polygon",
        instructions="Use market MCP tools and be explicit when a key is missing.",
        model="gpt-4o-mini",
        mcp_servers=[mcp_server],
    )
    result_polygon = await Runner.run(trader_agent, request_polygon)

display(Markdown(result_polygon.final_output))

The previous close for Apple Inc. (AAPL) is $246.63, which reflects the last trading price of the stock at the end of the previous trading day, serving as a benchmark for its performance in the upcoming session.

### Notes

- Use Stooq symbol format like `aapl.us`.
- Polygon call needs `POLYGON_API_KEY`.